### 1. Sales Summary (Daily KPI)

In [0]:
from pyspark.sql.functions import sum, countDistinct, avg

sales_summary = (
    spark.table("sales.gold.fact_sales")
    .groupBy("OrderDate")
    .agg(
        sum("SalesAmount").alias("total_revenue"),
        countDistinct("OrderID").alias("total_orders"),
        sum("Quantity").alias("total_quantity"),
        avg("SalesAmount").alias("avg_order_value")
    )
)

display(sales_summary)

### 2. Product Performance

In [0]:
from pyspark.sql.functions import sum

fact = spark.table("sales.gold.fact_sales")
product = spark.table("sales.gold.dim_product")

product_perf = (
    fact.join(product, "ProductID")
    .groupBy("ProductName", "Category")
    .agg(
        sum("SalesAmount").alias("total_sales"),
        sum("Quantity").alias("total_quantity")
    )
    .orderBy("total_sales", ascending=False)
)

display(product_perf)

### 3. Customer KPI

In [0]:
from pyspark.sql.functions import countDistinct, sum, avg

customer = spark.table("sales.gold.dim_customer")

customer_kpi = (
    fact.join(customer, "CustomerID")
    .groupBy("FullName")
    .agg(
        countDistinct("OrderID").alias("total_orders"),
        sum("SalesAmount").alias("total_spent"),
        avg("SalesAmount").alias("avg_order_value")
    )
    .orderBy("total_spent", ascending=False)
)

display(customer_kpi)

### 4. Store Performance

In [0]:
store = spark.table("sales.gold.dim_store")

store_perf = (
    fact.join(store, "StoreID")
    .groupBy("StoreName", "City")
    .agg(
        sum("SalesAmount").alias("total_sales"),
        countDistinct("OrderID").alias("total_orders")
    )
)

display(store_perf)

### 5. Delivery Performance

In [0]:
from pyspark.sql.functions import avg, sum, when

delivery_perf = (
    fact.agg(
        avg("delivery_days").alias("avg_delivery_time"),
        sum(when(fact.is_delivered == 1, 1).otherwise(0)).alias("delivered_orders"),
        sum(when(fact.is_delivered == 0, 1).otherwise(0)).alias("pending_orders")
    )
)

display(delivery_perf)

### 6. Payment Success Rate

In [0]:
from pyspark.sql.functions import sum, count

payment_kpi = (
    fact.agg(
        (sum(when(fact.is_success == 1, 1).otherwise(0)) * 100.0 / count("*"))
        .alias("success_rate")
    )
)

display(payment_kpi)

### 7. Monthly Sales Trend

In [0]:
from pyspark.sql.functions import year, month, sum

monthly_sales = (
    fact
    .withColumn("year", year("OrderDate"))
    .withColumn("month", month("OrderDate"))
    .groupBy("year", "month")
    .agg(sum("SalesAmount").alias("revenue"))
    .orderBy("year", "month")
)

display(monthly_sales)

In [0]:
sales_summary.write.mode("overwrite").saveAsTable("sales.gold.gold_sales_summary")

product_perf.write.mode("overwrite").saveAsTable("sales.gold.gold_product_performance")

customer_kpi.write.mode("overwrite").saveAsTable("sales.gold.gold_customer_kpi")

store_perf.write.mode("overwrite").saveAsTable("sales.gold.gold_store_performance")